In [1]:
def split_by_double_newline_from_file(file_path: str):
    # 读取 txt 文件内容
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # 按两个换行符切分，并清理首尾空格
    chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
    return chunks


file_path = "data/chatdoctor/chatdoctor.txt"  # 把这里改成你的 txt 文件路径
chunks = split_by_double_newline_from_file(file_path)

# for i, c in enumerate(chunks, 1):
#     print(f"Chunk {i} (长度 {len(c)}):\n{c}\n---")

In [2]:
len(chunks)

207408

In [10]:
chunks[0]

'input: i had what feels like a muscle cramp about an hour ago under the left bottom rib. it lasted about a minute and then went away. i had no other pains or dificulties since. could this have been a simptom of a minor heart attack. do heart attack symptoms come one at a time or are there more than one symptom when they occur?\noutput: No this is not a symptom of great attack.... it is normal after some stressful activity. No need to worry. If same thing happen again let me know'

In [5]:
import json

In [9]:
paths = ["data/chatdoctor/corpus.jsonl"]
corpus = []

for path in paths:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            corpus.append(doc.get("text", ""))

print(f"共加载 {len(corpus)} 条记录")
print("第一条：", corpus[0])

共加载 207408 条记录
第一条： input: i had what feels like a muscle cramp about an hour ago under the left bottom rib. it lasted about a minute and then went away. i had no other pains or dificulties since. could this have been a simptom of a minor heart attack. do heart attack symptoms come one at a time or are there more than one symptom when they occur?
output: No this is not a symptom of great attack.... it is normal after some stressful activity. No need to worry. If same thing happen again let me know


In [11]:
import numpy as np
import re

def analyze_chunks(chunks, name=""):
    print(f"\n===== 分析 {name} =====")
    
    # 1. 长度分布
    lengths = [len(c) for c in chunks]
    print(f"chunk 数量: {len(chunks)}")
    print(f"平均长度: {np.mean(lengths):.2f}, 中位数: {np.median(lengths)}, 标准差: {np.std(lengths):.2f}")
    print(f"最短: {min(lengths)}, 最长: {max(lengths)}")

    # 2. 边界特征
    boundary_tokens = [c[-1] if c else "" for c in chunks]
    end_punct_count = sum(1 for t in boundary_tokens if t in "。！？.?!")
    print(f"以句末标点结尾的 chunk 占比: {end_punct_count/len(chunks):.2%}")

    # 3. 重叠情况
    overlap_count = 0
    for i in range(len(chunks)-1):
        if chunks[i] and chunks[i+1]:
            # 检查相邻 chunk 是否有重叠（至少 10 个字符相同）
            overlap_len = min(len(chunks[i]), len(chunks[i+1]), 50)  # 限制比对长度
            if chunks[i][-overlap_len:] in chunks[i+1][:overlap_len]:
                overlap_count += 1
    print(f"相邻 chunk 有重叠的比例: {overlap_count/len(chunks):.2%}")


In [13]:
analyze_chunks(chunks, "方案A")
analyze_chunks(corpus, "方案B")


===== 分析 方案A =====
chunk 数量: 207408
平均长度: 912.25, 中位数: 841.0, 标准差: 381.36
最短: 60, 最长: 11770
以句末标点结尾的 chunk 占比: 68.63%
相邻 chunk 有重叠的比例: 0.00%

===== 分析 方案B =====
chunk 数量: 207408
平均长度: 912.25, 中位数: 841.0, 标准差: 381.36
最短: 60, 最长: 11770
以句末标点结尾的 chunk 占比: 68.63%
相邻 chunk 有重叠的比例: 0.00%
